In [9]:
import pandas as pd
import numpy as np
import os

# =========================================================
# SETTINGS
# =========================================================
DATA_PATH = r"C:\IDEAL_Programming\processed\final\IDEAL_final_hourly_dataset.csv"

HOME_ID = "home77"
SEASON_NAME = "winter"      # winter / spring / summer / autumn
MONTH_DAY = "01-28"         # 01-28 / 04-09 / 07-28 / 10-30

OUTPUT_DIR = r"C:\IDEAL_Programming\One_by_One\Home77"
HISTORY_HOURS = 48

# =========================================================
# LOAD
# =========================================================
df = pd.read_csv(DATA_PATH, low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["timestamp"]).copy()

df = df[df["home_id"] == HOME_ID].copy()
df = df.sort_values("timestamp").reset_index(drop=True)

df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["month_day"] = df["timestamp"].dt.strftime("%m-%d")

if df.empty:
    raise ValueError(f"No data found for home_id={HOME_ID}")

# =========================================================
# HELPERS
# =========================================================
def get_ui_target_timestamp(month_day, season_name):
    if season_name == "winter":
        year = 2027
    else:
        year = 2026
    return pd.Timestamp(f"{year}-{month_day} 00:00:00")


def format_ui_timestamp(ts):
    return f"{ts.month}/{ts.day}/{ts.year} {ts.hour}:{ts.minute:02d}"


def get_valid_target_days(home_df, month_day):
    temp = home_df[home_df["month_day"] == month_day].copy()

    if temp.empty:
        return pd.DataFrame(columns=["date"])

    counts = (
        temp.groupby("date")["hour"]
        .nunique()
        .reset_index(name="n_hours")
    )

    valid_days = counts[counts["n_hours"] == 24][["date"]].copy()
    return valid_days


def extract_history_window(home_df, target_date, history_hours=168):
    target_start = pd.Timestamp(target_date)
    history_start = target_start - pd.Timedelta(hours=history_hours)
    history_end = target_start - pd.Timedelta(hours=1)

    window = home_df[
        (home_df["timestamp"] >= history_start) &
        (home_df["timestamp"] <= history_end)
    ][["timestamp", "consumption_Wh"]].copy()

    if len(window) != history_hours:
        return None

    window = window.sort_values("timestamp").reset_index(drop=True)

    expected_index = pd.date_range(start=history_start, end=history_end, freq="h")
    if not window["timestamp"].equals(pd.Series(expected_index)):
        return None

    window["history_hour"] = np.arange(-history_hours, 0)
    window["target_date"] = pd.Timestamp(target_date)

    return window


# =========================================================
# FIND VALID TARGET DAYS
# =========================================================
valid_target_days = get_valid_target_days(df, MONTH_DAY)

if valid_target_days.empty:
    raise ValueError(
        f"No valid 24-hour target days found for {HOME_ID} on month_day={MONTH_DAY}"
    )

print("=" * 90)
print(f"HOME: {HOME_ID}")
print(f"SEASON: {SEASON_NAME}")
print(f"MONTH_DAY: {MONTH_DAY}")
print(f"Valid target days found: {len(valid_target_days)}")
print(valid_target_days)

# =========================================================
# EXTRACT ALL VALID 168h WINDOWS
# =========================================================
history_windows = []

for row in valid_target_days.itertuples(index=False):
    target_date = row.date
    hist = extract_history_window(df, target_date, history_hours=HISTORY_HOURS)

    if hist is not None:
        history_windows.append(hist)

if not history_windows:
    raise ValueError(
        f"Target days were found for {HOME_ID}, but no complete {HISTORY_HOURS}h history windows exist."
    )

all_hist = pd.concat(history_windows, ignore_index=True)

# =========================================================
# BUILD PROXY HISTORY
# mean across all valid years for same home and same relative hour
# =========================================================
proxy_history = (
    all_hist.groupby("history_hour", as_index=False)
    .agg(consumption_Wh=("consumption_Wh", "mean"))
    .sort_values("history_hour")
    .reset_index(drop=True)
)

# =========================================================
# BUILD UI TIMESTAMPS
# =========================================================
ui_target_ts = get_ui_target_timestamp(MONTH_DAY, SEASON_NAME)
ui_history_start = ui_target_ts - pd.Timedelta(hours=HISTORY_HOURS)

proxy_history["timestamp"] = pd.date_range(
    start=ui_history_start,
    periods=HISTORY_HOURS,
    freq="h"
)

# =========================================================
# UI-READY OUTPUT
# =========================================================
ui_ready_df = proxy_history[["timestamp", "consumption_Wh"]].copy()
ui_ready_df["timestamp"] = ui_ready_df["timestamp"].apply(format_ui_timestamp)

# =========================================================
# DETAILED OUTPUT
# =========================================================
detailed_df = proxy_history.copy()
detailed_df["home_id"] = HOME_ID
detailed_df["season"] = SEASON_NAME
detailed_df["month_day"] = MONTH_DAY
detailed_df["history_hours"] = HISTORY_HOURS

detailed_df = detailed_df[[
    "home_id", "season", "month_day", "history_hours",
    "history_hour", "timestamp", "consumption_Wh"
]]

# =========================================================
# USED TARGET DATES OUTPUT
# =========================================================
used_target_dates_df = pd.DataFrame({
    "home_id": HOME_ID,
    "target_date": sorted(pd.to_datetime(all_hist["target_date"]).dt.date.unique())
})

# =========================================================
# SAVE
# =========================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name = f"{HOME_ID}_{SEASON_NAME}_history_168h"

ui_path = os.path.join(OUTPUT_DIR, f"{base_name}_ui.csv")
detailed_path = os.path.join(OUTPUT_DIR, f"{base_name}_detailed.csv")
used_dates_path = os.path.join(OUTPUT_DIR, f"{base_name}_used_target_dates.csv")

ui_ready_df.to_csv(ui_path, index=False)
detailed_df.to_csv(detailed_path, index=False)
used_target_dates_df.to_csv(used_dates_path, index=False)

# =========================================================
# PRINT
# =========================================================
print("\n" + "=" * 90)
print("FILES SAVED")
print("=" * 90)
print(ui_path)
print(detailed_path)
print(used_dates_path)

print("\nUI-READY HISTORY PREVIEW:")
print(ui_ready_df.head())
print("...")
print(ui_ready_df.tail())

print(f"\nTotal rows in UI history: {len(ui_ready_df)}")
print(f"Expected rows: {HISTORY_HOURS}")

HOME: home77
SEASON: winter
MONTH_DAY: 01-28
Valid target days found: 1
         date
0  2017-01-28

FILES SAVED
C:\IDEAL_Programming\One_by_One\Home77\home77_winter_history_168h_ui.csv
C:\IDEAL_Programming\One_by_One\Home77\home77_winter_history_168h_detailed.csv
C:\IDEAL_Programming\One_by_One\Home77\home77_winter_history_168h_used_target_dates.csv

UI-READY HISTORY PREVIEW:
        timestamp  consumption_Wh
0  1/26/2027 0:00           167.0
1  1/26/2027 1:00            84.0
2  1/26/2027 2:00            94.0
3  1/26/2027 3:00           143.0
4  1/26/2027 4:00           138.0
...
          timestamp  consumption_Wh
43  1/27/2027 19:00           528.0
44  1/27/2027 20:00           479.0
45  1/27/2027 21:00           478.0
46  1/27/2027 22:00           478.0
47  1/27/2027 23:00           305.0

Total rows in UI history: 48
Expected rows: 48
